In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv')

In [4]:
df

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0
...,...,...,...,...,...,...,...,...,...,...,...
9995,2005,USA,Gasoline,All-wheel drive,4,2470,6,271.0,4690,18.9,27.4
9996,1993,Asia,Gasoline,All-wheel drive,2,2300,6,244.0,4370,19.0,30.0
9997,2014,USA,Gasoline,All-wheel drive,3,2480,6,267.0,4590,18.4,28.1
9998,2004,USA,Hybrid,Rear-wheel drive,5,2180,6,270.0,4450,20.9,32.2


In [5]:
df.isnull().sum()

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

In [6]:
df.horsepower.median()

np.float64(254.0)

In [7]:
df.columns

Index(['model_year', 'origin', 'fuel_type', 'drivetrain', 'num_doors',
       'engine_displacement', 'num_cylinders', 'horsepower', 'vehicle_weight',
       'acceleration', 'fuel_efficiency_mpg'],
      dtype='str')

In [12]:
df.dtypes

model_year               int64
origin                     str
fuel_type                  str
drivetrain                 str
num_doors                int64
engine_displacement      int64
num_cylinders            int64
horsepower             float64
vehicle_weight           int64
acceleration           float64
fuel_efficiency_mpg    float64
dtype: object

In [13]:
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [14]:
import numpy as np
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv')

In [15]:
df.isna().sum()

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

In [21]:
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

In [22]:
idx = np.arange(n)
np.random.seed(42)
np.random.shuffle(idx)

In [23]:
df_shuffled = df.iloc[idx]
df_train = df_shuffled.iloc[:n_train].reset_index(drop=True)
df_val = df_shuffled.iloc[n_train:n_train + n_val].reset_index(drop=True)

In [24]:
df_train = df.iloc[idx[:n_train]].reset_index(drop=True)
df_val   = df.iloc[idx[n_train:n_train + n_val]].reset_index(drop=True)
df_test  = df.iloc[idx[n_train + n_val:]].reset_index(drop=True)

In [15]:
features = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']

In [38]:
mean_hp = df_train['horsepower'].mean()

In [17]:
mean_hp

np.float64(254.46338337605272)

In [18]:
def prepare_X(df, fill_value):
    df= df.copy()
    return df[features].fillna(fill_value).values 

In [25]:
def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)
    return w_full[0], w_full[1:]

In [26]:
def rmse(y, y_pred):
    error = y_pred - y
    mse = (error ** 2).mean()
    return np.sqrt(mse)

In [27]:
#fill 0

In [28]:
X_train = prepare_X(df_train, 0)
X_val = prepare_X(df_val, 0)

In [40]:
w0, w = train_linear_regression(X_train, y_train)
y_pred = w0 + X_val.dot(w)
rmse_0 = round(rmse(y_val, y_pred), 3)

In [41]:
rmse_0

np.float64(0.0)

In [31]:
#fill mean

In [42]:
X_train = prepare_X(df_train, mean_hp)
X_val = prepare_X(df_val, mean_hp)

In [43]:
w0, w = train_linear_regression(X_train, y_train)
y_pred = w0 + X_val.dot(w)
rmse_mean = round(rmse(y_val, y_pred), 3) 

In [44]:
rmse_mean

np.float64(2.202)

In [35]:
#rmse_mean < rmse_0

In [27]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols.remove('fuel_efficiency_mpg') 

In [28]:
y_train = df_train['fuel_efficiency_mpg'].values
y_val   = df_val['fuel_efficiency_mpg'].values
y_test  = df_test['fuel_efficiency_mpg'].values

In [29]:
X_train = df_train[numeric_cols].fillna(0).values.astype(float)
X_val   = df_val[numeric_cols].fillna(0).values.astype(float)
X_test  = df_test[numeric_cols].fillna(0).values.astype(float)

In [31]:
def train_linear_regression_reg(X, y, r=0.0):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])     # add r to the diagonal

    w = np.linalg.solve(XTX, X.T.dot(y))     # stable solve (no explicit inverse)
    return w[0], w[1:]  

In [32]:
def rmse(y, y_pred):
    return np.sqrt(np.mean((y - y_pred) ** 2))

In [33]:
for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    w0, w = train_linear_regression_reg(X_train, y_train, r=r)
    y_pred = w0 + X_val.dot(w)
    score = round(rmse(y_val, y_pred), 4)
    print(f'r={r:<6} rmse={score}')

r=0      rmse=2.2047
r=0.01   rmse=2.2052
r=0.1    rmse=2.2233
r=1      rmse=2.3469
r=5      rmse=2.4062
r=10     rmse=2.4162
r=100    rmse=2.4257


In [34]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols.remove('fuel_efficiency_mpg')

In [35]:
def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    XTX = X.T.dot(X)
    w = np.linalg.solve(XTX, X.T.dot(y))
    return w[0], w[1:]

In [36]:
def rmse(y, y_pred):
    return np.sqrt(np.mean((y - y_pred) ** 2))

In [37]:
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

In [38]:
scores = []

for seed in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]:
    idx = np.arange(n)
    np.random.seed(seed)
    np.random.shuffle(idx)

    df_train = df.iloc[idx[:n_train]].reset_index(drop=True)
    df_val   = df.iloc[idx[n_train:n_train + n_val]].reset_index(drop=True)
    # df_test not needed for scoring here

    y_train = df_train['fuel_efficiency_mpg'].values
    y_val   = df_val['fuel_efficiency_mpg'].values

    X_train = df_train[numeric_cols].fillna(0).values.astype(float)
    X_val   = df_val[numeric_cols].fillna(0).values.astype(float)

    w0, w = train_linear_regression(X_train, y_train)
    y_pred = w0 + X_val.dot(w)
    score = rmse(y_val, y_pred)
    scores.append(score)
    print(f'seed={seed} rmse={round(score, 4)}')

std = np.std(scores)
print('std:', round(std, 3))

seed=0 rmse=2.2413
seed=1 rmse=2.2019
seed=2 rmse=2.1589
seed=3 rmse=2.2004
seed=4 rmse=2.1971
seed=5 rmse=2.2457
seed=6 rmse=2.2696
seed=7 rmse=2.1839
seed=8 rmse=2.2143
seed=9 rmse=2.2053
std: 0.031


In [39]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols.remove('fuel_efficiency_mpg')

In [40]:
def train_linear_regression_reg(X, y, r=0.0):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])
    w = np.linalg.solve(XTX, X.T.dot(y))
    return w[0], w[1:]

In [41]:
def rmse(y, y_pred):
    return np.sqrt(np.mean((y - y_pred) ** 2))

In [42]:
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

idx = np.arange(n)
np.random.seed(9)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]].reset_index(drop=True)
df_val   = df.iloc[idx[n_train:n_train + n_val]].reset_index(drop=True)
df_test  = df.iloc[idx[n_train + n_val:]].reset_index(drop=True)

In [43]:
df_full_train = pd.concat([df_train, df_val]).reset_index(drop=True)

y_full_train = df_full_train['fuel_efficiency_mpg'].values
y_test       = df_test['fuel_efficiency_mpg'].values

X_full_train = df_full_train[numeric_cols].fillna(0).values.astype(float)
X_test       = df_test[numeric_cols].fillna(0).values.astype(float)

In [44]:
w0, w = train_linear_regression_reg(X_full_train, y_full_train, r=0.001)
y_pred = w0 + X_test.dot(w)

score = rmse(y_test, y_pred)
print('Test RMSE:', round(score, 4))

Test RMSE: 2.2346
